# Line Segmentation of the manual-transcription pages (DocLayout-YOLO → Kraken, local)

Local version (data lives in the repo under `transcriptions/`). The Colab/Drive twin is
`segmentManualTranscriptionsIntoLinesWithKrakenFromGoogleColabDrive.ipynb`; this one differs only
in paths (repo-relative, no Drive mount). Kraken runs on CPU; a GPU just makes it faster.

```
transcriptions/
  5/  B5_P_014.txt  B5_P_015.txt  ...     # manual transcriptions (one physical line per text line)
  6/  B6_P_014.txt  ...
  images/5/  B5_P_014.jpg  B5_P_015.jpg  ...   # page scans (same basename as the .txt)
  images/6/  B6_P_014.jpg  ...
```

We have page images + ordered text but **no line coordinates**, so to build `(line_image, line_text)`
pairs we must **detect** each line region and **align** it to the right transcription line.

**Two-stage detection pipeline:**
1. **DocLayout-YOLO (DocStructBench)** — a document layout detector, run first to find **content
   regions** (`plain text`, `title`) and ignore the rest. *Caveat:* on these handwritten scans it
   tends to label the whole page as `figure`, so it may find no content region and effectively
   skip gating — set `USE_LAYOUT_FILTER=False` to drop the dependency entirely.
2. **Kraken baseline segmentation (`blla`)** — predicts a baseline + **boundary polygon** per line
   (handles slanted 18th-c. cursive) in reading order.

**Detected lines usually exceed the transcription** because Kraken splits margin date labels,
right-aligned citations and interlinear words off the writing line, whereas the transcription folds
them into one line. An **automatic same-line merge** (by vertical overlap, `AUTO_MERGE_VOVERLAP`)
recombines those first; you then fix any remainder with the per-page `drop`/`merge` dict.

**Tight crops via polygon masking.** A line's axis-aligned bounding box scoops up ascenders /
descenders from the lines above and below. To avoid that we crop the box but **mask every pixel
outside the line's boundary polygon to white** (`MASK_TO_POLYGON`), and keep vertical padding small
(`PAD_Y`). The result is just the target line on a clean background.

**Alignment — verify & correct:** overlay numbered boxes on each page, print numbered ground-truth
lines, flag pages where `#boxes ≠ #text-lines`, fix them, and only export pages whose counts match.

In [ ]:
# Kraken = line baselines; doclayout-yolo = layout regions; huggingface_hub fetches the YOLO weights.
# If these pull incompatible pins into your env, install them in a dedicated venv.
# (Set USE_LAYOUT_FILTER=False below to skip doclayout-yolo + huggingface_hub.)
!pip -q install kraken doclayout-yolo huggingface_hub

In [ ]:
import os
import glob
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Polygon
from PIL import Image, ImageDraw, ImageFilter

from kraken import blla
from kraken.lib import vgsl

plt.rcParams["figure.figsize"] = (10, 14)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# Config & page pairing
Transcription `transcriptions/<F>/<stem>.txt` pairs with the same-named image `transcriptions/images/<F>/<stem>.jpg` (e.g. `5/B5_P_014.txt` ↔ `images/5/B5_P_014.jpg`).

In [ ]:
# --- paths (relative to the repo root; this notebook lives in notebooks/) ---
REPO_ROOT    = os.path.abspath("..")
TRANSCR_ROOT = os.path.join(REPO_ROOT, "transcriptions")
IMAGES_ROOT  = os.path.join(TRANSCR_ROOT, "images")
FOLDERS      = ["5", "6"]                     # source sub-folders to process

# --- outputs ---
OUT_DIR  = os.path.join(TRANSCR_ROOT, "line_crops")   # cropped line images, per folder
CSV_PATH = os.path.join(TRANSCR_ROOT, "lines.csv")    # file_name,text  (LineOCRDataset format)

# --- cropping knobs ---
PAD_X = 6      # horizontal padding around the line box (px)
PAD_Y = 2      # vertical padding — keep small to avoid neighbour-line bleed (px)
MASK_TO_POLYGON = True   # white-out everything outside the line's boundary polygon
POLY_PAD = 2   # dilate the polygon mask by this many px so the line's own strokes aren't clipped

# --- box filtering ---
MIN_W = 40     # drop detected boxes narrower than this (noise: page numbers, stray marks)
MIN_H = 12     # drop detected boxes shorter than this

# --- automatic same-line merge ---
# Detected > ground truth is almost always Kraken splitting ONE written line into several boxes
# (left-margin date labels, right-aligned citations, interlinear/superscript words). Your
# transcription folds those into one line, so we re-merge boxes that share most of their vertical
# extent (i.e. sit on the same writing band) before any manual correction.
AUTO_MERGE_VOVERLAP = False  # off: keep every Kraken line separate (set True to merge same-band splits)
VOVERLAP_FRAC = 0.5          # merge if overlap >= this fraction of the shorter box's height

# --- DocLayout-YOLO (DocStructBench) pre-filter ---
# NOTE: on these handwritten scans DocStructBench labels the whole page as `figure`, so with the
# default CONTENT_CLASSES it finds 0 content regions and does nothing. Add "figure" to gate by the
# detected writing block, or set USE_LAYOUT_FILTER=False to skip it (and its imports) entirely.
USE_LAYOUT_FILTER = True
LAYOUT_REPO   = "juliozhao/DocLayout-YOLO-DocStructBench"
LAYOUT_FILE   = "doclayout_yolo_docstructbench_imgsz1024.pt"
LAYOUT_IMGSZ  = 1024
LAYOUT_CONF   = 0.2
CONTENT_CLASSES = {"plain text", "title"}
REGION_PAD    = 20         # px slack when testing whether a Kraken line falls inside a region

def gt_lines_for(txt_path):
    """Transcription lines, stripped, with blank lines removed — in reading order."""
    with open(txt_path, encoding="utf-8") as f:
        return [ln.strip() for ln in f.read().splitlines() if ln.strip()]

pages = []
for folder in FOLDERS:
    for txt_path in sorted(glob.glob(os.path.join(TRANSCR_ROOT, folder, "*.txt"))):
        stem     = os.path.splitext(os.path.basename(txt_path))[0]   # e.g. B5_P_014
        img_name = stem + ".jpg"                                       # B5_P_014 -> B5_P_014.jpg
        img_path = os.path.join(IMAGES_ROOT, folder, img_name)
        if not os.path.exists(img_path):
            print(f"  !! missing image for {stem}: {img_path}")
            continue
        pages.append({
            "key":      f"{folder}/{stem}",
            "folder":   folder,
            "stem":     stem,
            "img_path": img_path,
            "gt":       gt_lines_for(txt_path),
        })

print(f"Paired {len(pages)} pages:")
for p in pages:
    print(f"  {p['key']:14s}  gt_lines={len(p['gt'])}")

# Stage 1 — DocLayout-YOLO: detect content regions
Skipped cleanly if `USE_LAYOUT_FILTER=False`. Keeps boxes of the `CONTENT_CLASSES` as **content
regions**; Kraken's line boxes are later gated by these.

In [ ]:
CONTENT_REGIONS, LAYOUT_DETS = {}, {}

if USE_LAYOUT_FILTER:
    from doclayout_yolo import YOLOv10
    from huggingface_hub import hf_hub_download

    layout_path  = hf_hub_download(repo_id=LAYOUT_REPO, filename=LAYOUT_FILE)
    layout_model = YOLOv10(layout_path)
    print("DocLayout-YOLO weights:", layout_path)

    def detect_regions(img_path):
        res = layout_model.predict(img_path, imgsz=LAYOUT_IMGSZ, conf=LAYOUT_CONF,
                                   device=device, verbose=False)[0]
        names = res.names
        xyxy  = res.boxes.xyxy.cpu().numpy()
        clss  = res.boxes.cls.cpu().numpy()
        confs = res.boxes.conf.cpu().numpy()
        regions, dets = [], []
        for (x0, y0, x1, y1), c, cf in zip(xyxy, clss, confs):
            name = names[int(c)]
            box  = (float(x0), float(y0), float(x1), float(y1))
            dets.append((name, float(cf), box))
            if name in CONTENT_CLASSES:
                regions.append(box)
        return regions, dets

    for p in pages:
        regions, dets = detect_regions(p["img_path"])
        CONTENT_REGIONS[p["key"]] = regions
        LAYOUT_DETS[p["key"]]     = dets
        print(f"  {p['key']:14s}  content regions: {len(regions)}/{len(dets)} dets   "
              f"classes={sorted({d[0] for d in dets})}")
        if not regions:
            print(f"      (no content region — Kraken lines for this page will NOT be gated)")
else:
    print("USE_LAYOUT_FILTER=False — skipping DocLayout-YOLO (no region gating).")

In [ ]:
# Inspect DocLayout-YOLO output (only if it ran): green solid = kept content, orange dashed = ignored.
def show_layout(p):
    im = Image.open(p["img_path"]).convert("RGB")
    fig, ax = plt.subplots(figsize=(10, 14))
    ax.imshow(im)
    for name, cf, (x0, y0, x1, y1) in LAYOUT_DETS.get(p["key"], []):
        keep = name in CONTENT_CLASSES
        ax.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False,
                               edgecolor="lime" if keep else "orange",
                               linewidth=2, linestyle="-" if keep else "--"))
        ax.text(x0, y0 - 4, f"{name} {cf:.2f}",
                color="lime" if keep else "orange", fontsize=9, weight="bold")
    ax.set_title(f"{p['key']} — DocLayout-YOLO (green=kept content, orange=ignored)", fontsize=12)
    ax.axis("off")
    plt.tight_layout()
    plt.show()

if USE_LAYOUT_FILTER and LAYOUT_DETS:
    for p in pages:
        show_layout(p)
else:
    print("Nothing to show (layout filter disabled or no detections).")

# Stage 2 — Kraken baseline segmentation
Segmentation is the slow step, so we run it once and cache each line's **bounding box _and_ boundary
polygon** per page. The polygon is what lets us crop tightly (mask out neighbouring lines) later.
Re-running the verify/correct cells is then instant (no re-segmentation).

In [ ]:
def load_seg_model():
    """Load Kraken's bundled default baseline-segmentation model (blla.mlmodel)."""
    import kraken
    cand = os.path.join(os.path.dirname(kraken.__file__), "blla.mlmodel")
    if os.path.exists(cand):
        print("Loading bundled segmentation model:", cand)
        return vgsl.TorchVGSLModel.load_model(cand)
    print("Bundled model not found — falling back to blla.segment default.")
    return None

seg_model = load_seg_model()

def boundary_of(line):
    """Return a line's boundary polygon as a list of (x, y), across kraken API versions."""
    b = getattr(line, "boundary", None)
    if b is None and isinstance(line, dict):
        b = line.get("boundary")
    return b

def segment_page(img_path):
    """Run Kraken baseline segmentation; return line records [{'bbox':(x0,y0,x1,y1),
    'polys':[[(x,y),...]]}, ...] — bbox for geometry, polys for tight masked cropping."""
    im = Image.open(img_path).convert("RGB")
    try:
        seg = blla.segment(im, model=seg_model, device=device)
    except TypeError:
        seg = blla.segment(im, model=seg_model) if seg_model is not None else blla.segment(im)
    lines = getattr(seg, "lines", None)
    if lines is None and isinstance(seg, dict):
        lines = seg.get("lines", [])

    records = []
    for ln in lines:
        poly = boundary_of(ln)
        if not poly:
            continue
        pts = [(float(pt[0]), float(pt[1])) for pt in poly]
        xs = [pt[0] for pt in pts]
        ys = [pt[1] for pt in pts]
        records.append({"bbox": (min(xs), min(ys), max(xs), max(ys)), "polys": [pts]})
    return records

RAW_LINES = {}
for p in pages:
    RAW_LINES[p["key"]] = segment_page(p["img_path"])
    print(f"  {p['key']:14s}  raw lines: {len(RAW_LINES[p['key']]):3d}   gt lines: {len(p['gt'])}")

# Per-page correction dict
The boxes are already **auto-merged by vertical overlap** (`AUTO_MERGE_VOVERLAP`), which collapses
most of the over-detection. Use this dict only for what's left. Indices refer to the **auto-merged**
lines shown in the overlay (the numbers drawn on each page).

- `drop`:  list of line indices to remove (noise the layout filter / size filter didn't catch).
- `merge`: list of index-groups to fuse into one line — for splits the auto-merge missed. Merged
  lines keep **both** polygons, so the masked crop covers each piece.

Leave a page out of the dict to use its auto-merged lines unchanged. Re-run the overlay cell after
editing. If a page is still over-detected everywhere, lower `VOVERLAP_FRAC` (e.g. 0.4); if distinct
lines are wrongly merged, raise it (e.g. 0.6).

In [ ]:
# Example:
#   "6/B6_P_014": {"drop": [0], "merge": [[8, 9]]}
PAGE_ADJUSTMENTS = {
    # "<folder>/<stem>": {"drop": [...], "merge": [[...], ...]},
}

In [ ]:
def _union_bbox(recs):
    bs = [r["bbox"] for r in recs]
    return (min(b[0] for b in bs), min(b[1] for b in bs),
            max(b[2] for b in bs), max(b[3] for b in bs))

def _merge_recs(recs):
    """Fuse several line records into one: union the bbox, concatenate the polygons."""
    if len(recs) == 1:
        return recs[0]
    return {"bbox": _union_bbox(recs),
            "polys": [poly for r in recs for poly in r["polys"]]}

def _center_in_regions(bbox, regions, pad):
    cx, cy = (bbox[0] + bbox[2]) / 2, (bbox[1] + bbox[3]) / 2
    for rx0, ry0, rx1, ry1 in regions:
        if rx0 - pad <= cx <= rx1 + pad and ry0 - pad <= cy <= ry1 + pad:
            return True
    return False

def auto_merge_vertical(recs, frac):
    """Greedily merge boxes that sit on the same writing band. Walking top-to-bottom, a box joins
    the current line if its vertical range overlaps that line's by >= `frac` of the shorter box's
    height — true for margin labels / citations / interlinear words split off one line, but not for
    the next genuine line below. Collapses horizontal splits without hand-merging."""
    if not recs:
        return recs
    groups = [[recs[0]]]
    gy0, gy1 = recs[0]["bbox"][1], recs[0]["bbox"][3]
    for r in recs[1:]:
        y0, y1 = r["bbox"][1], r["bbox"][3]
        overlap = min(gy1, y1) - max(gy0, y0)
        shorter = min(gy1 - gy0, y1 - y0)
        if shorter > 0 and overlap / shorter >= frac:
            groups[-1].append(r)
            gy0, gy1 = min(gy0, y0), max(gy1, y1)
        else:
            groups.append([r])
            gy0, gy1 = y0, y1
    return [_merge_recs(g) for g in groups]

def filtered_sorted_lines(key):
    """Raw Kraken line records -> drop tiny ones -> gate by DocLayout-YOLO content regions ->
    sort top-to-bottom -> auto-merge same-band splits. This is the indexing the PAGE_ADJUSTMENTS
    `drop`/`merge` indices refer to."""
    recs    = [r for r in RAW_LINES[key]
               if (r["bbox"][2] - r["bbox"][0]) >= MIN_W and (r["bbox"][3] - r["bbox"][1]) >= MIN_H]
    regions = CONTENT_REGIONS.get(key, [])
    if USE_LAYOUT_FILTER and regions:
        recs = [r for r in recs if _center_in_regions(r["bbox"], regions, REGION_PAD)]
    recs = sorted(recs, key=lambda r: (r["bbox"][1] + r["bbox"][3]) / 2)
    if AUTO_MERGE_VOVERLAP:
        recs = auto_merge_vertical(recs, VOVERLAP_FRAC)
    return recs

def processed_lines(key):
    """Apply PAGE_ADJUSTMENTS (drop + merge) to the filtered/sorted line records, preserving order."""
    recs = filtered_sorted_lines(key)
    adj  = PAGE_ADJUSTMENTS.get(key, {})
    drop = set(adj.get("drop", []))
    out  = []  # (rep_index, record)
    used = set()
    for group in adj.get("merge", []):
        grp = [recs[i] for i in group if i < len(recs) and i not in drop]
        if grp:
            out.append((min(group), _merge_recs(grp)))
            used.update(group)
    for i, r in enumerate(recs):
        if i in drop or i in used:
            continue
        out.append((i, r))
    out.sort(key=lambda t: t[0])
    return [r for _, r in out]

# Verify: overlay detected lines + numbered ground-truth lines
Green title = line count matches the transcription. Red = mismatch → fix it in `PAGE_ADJUSTMENTS`
above and re-run this cell. Red box (numbered) = the line's bbox; cyan outline = the boundary
polygon the crop is masked to; faint green dashed = the DocLayout-YOLO content region (if any). The
numbers are the indices to use in `drop`/`merge`.

In [ ]:
def review_page(p):
    key = p["key"]
    fln = filtered_sorted_lines(key)   # indices for drop/merge
    n_proc = len(processed_lines(key)) # final count after adjustments
    n_gt = len(p["gt"])
    ok = n_proc == n_gt

    im = Image.open(p["img_path"]).convert("RGB")
    fig, ax = plt.subplots(figsize=(10, 14))
    ax.imshow(im)
    for rx0, ry0, rx1, ry1 in CONTENT_REGIONS.get(key, []):
        ax.add_patch(Rectangle((rx0, ry0), rx1 - rx0, ry1 - ry0, fill=False,
                               edgecolor="lime", linewidth=1.5, linestyle="--", alpha=0.6))
    for i, r in enumerate(fln):
        x0, y0, x1, y1 = r["bbox"]
        ax.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0,
                               fill=False, edgecolor="red", linewidth=1.0))
        for poly in r["polys"]:
            ax.add_patch(Polygon(poly, closed=True, fill=False,
                                 edgecolor="cyan", linewidth=1.0, alpha=0.9))
        ax.text(x0 - 4, (y0 + y1) / 2, str(i), color="blue", fontsize=11,
                ha="right", va="center", weight="bold")
    ax.set_title(f"{key}   detected(after adj)={n_proc}  vs  gt={n_gt}   "
                 f"[{'OK' if ok else 'MISMATCH'}]",
                 color="green" if ok else "crimson", fontsize=12)
    ax.axis("off")
    plt.tight_layout()
    plt.show()

    print(f"--- {key}: {n_gt} ground-truth lines ---")
    for i, t in enumerate(p["gt"]):
        print(f"  {i:2d}: {t}")
    print()

for p in pages:
    review_page(p)

In [ ]:
# Status summary — which pages are ready to export.
ready, blocked = [], []
for p in pages:
    (ready if len(processed_lines(p["key"])) == len(p["gt"]) else blocked).append(p["key"])
print(f"READY  ({len(ready)}): {ready}")
print(f"NEEDS FIX ({len(blocked)}): {blocked}")
if blocked:
    print("\nEdit PAGE_ADJUSTMENTS for the NEEDS FIX pages, re-run the overlay cell, then re-run this.")

# Crop helper — tight, polygon-masked line images
Crops the padded bbox, then (if `MASK_TO_POLYGON`) whites-out everything outside the line's
boundary polygon — removing ascenders/descenders that bleed in from the neighbouring lines.

In [ ]:
def crop_line(im, rec):
    """Return a tight line-image crop for one record, masked to its polygon(s)."""
    W, H = im.size
    x0, y0, x1, y1 = rec["bbox"]
    cx0 = max(0, int(x0) - PAD_X); cy0 = max(0, int(y0) - PAD_Y)
    cx1 = min(W, int(x1) + PAD_X); cy1 = min(H, int(y1) + PAD_Y)
    crop = im.crop((cx0, cy0, cx1, cy1))

    if not MASK_TO_POLYGON or not rec.get("polys"):
        return crop

    mask = Image.new("L", crop.size, 0)
    draw = ImageDraw.Draw(mask)
    for poly in rec["polys"]:
        draw.polygon([(px - cx0, py - cy0) for px, py in poly], fill=255)
    if POLY_PAD > 0:                       # dilate so the line's own strokes aren't clipped
        mask = mask.filter(ImageFilter.MaxFilter(POLY_PAD * 2 + 1))
    white = Image.new("RGB", crop.size, (255, 255, 255))
    return Image.composite(crop, white, mask)

# Preview: bbox crop (left) vs polygon-masked crop (right) for a few lines of the first ready page.
_preview_key = next((p["key"] for p in pages
                     if len(processed_lines(p["key"])) == len(p["gt"])), pages[0]["key"])
_pv = next(p for p in pages if p["key"] == _preview_key)
_im = Image.open(_pv["img_path"]).convert("RGB")
_recs = processed_lines(_preview_key)[:6]
fig, axes = plt.subplots(len(_recs), 2, figsize=(14, 1.6 * len(_recs)))
for row, r in zip(np.atleast_2d(axes), _recs):
    x0, y0, x1, y1 = r["bbox"]
    raw = _im.crop((max(0, int(x0) - PAD_X), max(0, int(y0) - PAD_Y),
                    int(x1) + PAD_X, int(y1) + PAD_Y))
    row[0].imshow(raw);             row[0].set_title("bbox crop", fontsize=9);  row[0].axis("off")
    row[1].imshow(crop_line(_im, r)); row[1].set_title("polygon-masked", fontsize=9); row[1].axis("off")
plt.suptitle(f"Crop comparison — {_preview_key}", y=1.0)
plt.tight_layout()
plt.show()

# Export: crop the line images + write `lines.csv`
Only pages whose line count matches the transcription are exported (set `FORCE_ALL=True` to export
every page regardless — not recommended). Lines are paired to transcription lines **in reading order**.

In [ ]:
FORCE_ALL = False

os.makedirs(OUT_DIR, exist_ok=True)
rows, skipped = [], []

for p in pages:
    key, folder, stem = p["key"], p["folder"], p["stem"]
    recs = processed_lines(key)
    gt = p["gt"]
    if len(recs) != len(gt) and not FORCE_ALL:
        skipped.append(key)
        continue

    im = Image.open(p["img_path"]).convert("RGB")
    out_folder = os.path.join(OUT_DIR, folder)
    os.makedirs(out_folder, exist_ok=True)

    n = min(len(recs), len(gt))   # FORCE_ALL safety: pair as far as both go
    for i in range(n):
        crop_path = os.path.join(out_folder, f"{stem}_l{i:02d}.png")
        crop_line(im, recs[i]).save(crop_path)
        rows.append({"file_name": crop_path, "text": gt[i]})

df = pd.DataFrame(rows).reset_index(drop=True)
df.to_csv(CSV_PATH, index=False)
print(f"Wrote {len(df)} line pairs")
print("Crops dir:", OUT_DIR)
print("CSV      :", CSV_PATH)
if skipped:
    print("SKIPPED (count mismatch — fix PAGE_ADJUSTMENTS or set FORCE_ALL):", skipped)
df.head()

# Sanity-check the exported pairs

In [ ]:
N_SHOW = 12
if len(df):
    sample = df.sample(min(N_SHOW, len(df)), random_state=42).reset_index(drop=True)
    fig, axes = plt.subplots(len(sample), 1, figsize=(14, 1.8 * len(sample)))
    if len(sample) == 1:
        axes = [axes]
    for ax, row in zip(axes, sample.itertuples()):
        ax.imshow(Image.open(row.file_name), cmap="gray")
        ax.set_title(f"GT: {row.text}", fontsize=10, loc="left")
        ax.axis("off")
    plt.suptitle("Exported line crops with their transcription", y=1.0)
    plt.tight_layout()
    plt.show()

    char_lens = df["text"].str.len()
    print(f"Char length — min {char_lens.min()}, median {int(char_lens.median())}, "
          f"mean {char_lens.mean():.1f}, max {char_lens.max()}")
else:
    print("No pairs exported yet — resolve the mismatched pages first.")

# Using these pairs for training

`lines.csv` is in the `file_name,text` format the `LineOCRDataset` (in the fine-tune notebooks)
expects. To train on Colab:

1. Upload `transcriptions/line_crops/` **and** `lines.csv` to Drive.
2. In a fine-tune notebook, load the CSV and re-root `file_name` to the Drive path:

```python
df = pd.read_csv("/content/drive/MyDrive/.../lines.csv")
df["file_name"] = df["file_name"].apply(
    lambda p: os.path.join(CROPS_ROOT, os.path.relpath(p, LOCAL_CROPS_ROOT)))
train_dataset = LineOCRDataset(df, processor, train_transforms, MAX_TARGET_LENGTH)
```

With only ~10 pages this set is small — best used to **continue fine-tuning** an existing checkpoint
(or held out as an in-domain validation/test set), not to train from scratch.